<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Temperature_Celsius/Temperature_Online_Learning_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install river --quiet

In [ ]:
import pandas as pd
import numpy as np

from river import metrics, compose, preprocessing

print("River imported successfully")

River imported successfully


In [ ]:
path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/final_temperature_dataset.csv"
temp_df = pd.read_csv(path)

print("Dataset shape:", temp_df.shape)
temp_df.head()

Dataset shape: (130003, 16)


,uv_index,latitude,humidity,pressure_mb,air_quality_Ozone,condition_text,air_quality_Nitrogen_dioxide,cloud,visibility_km,longitude,air_quality_PM10,wind_mph,gust_mph,wind_degree,precip_mm,temperature_celsius
0,1.0,46.60,58,1012.0,62.2,2,2.5,0,16.0,-120.49,7.1,4.3,10.3,220,0.00,16.1
1,1.0,14.10,78,1017.0,23.3,32,3.7,37,10.0,-87.22,25.3,3.8,7.0,240,0.28,23.0
2,1.0,13.71,94,1010.0,5.9,23,7.7,50,10.0,-89.20,28.1,2.2,2.8,182,0.30,26.0
3,1.0,14.62,88,1019.0,0.4,19,35.0,100,5.0,-90.53,178.1,13.6,18.1,190,0.09,20.0
4,1.0,17.25,89,1007.0,34.0,30,0.3,94,10.0,-88.77,32.1,4.3,6.5,99,0.00,26.0


In [ ]:
target = "temperature_celsius"

In [ ]:
# -----------------------------
# FEATURE ENGINEERING FOR TEMPERATURE
# -----------------------------
temp_df["temp_lag1"] = temp_df[target].shift(1)
temp_df["temp_lag2"] = temp_df[target].shift(2)
temp_df["temp_lag3"] = temp_df[target].shift(3)

temp_df["temp_roll3_mean"] = temp_df[target].rolling(window=3).mean()
temp_df["temp_roll5_mean"] = temp_df[target].rolling(window=5).mean()

if "humidity" in temp_df.columns:
    temp_df["humidity_lag1"] = temp_df["humidity"].shift(1)

if "pressure_mb" in temp_df.columns:
    temp_df["pressure_lag1"] = temp_df["pressure_mb"].shift(1)

if "wind_mph" in temp_df.columns:
    temp_df["wind_lag1"] = temp_df["wind_mph"].shift(1)

if "cloud" in temp_df.columns:
    temp_df["cloud_lag1"] = temp_df["cloud"].shift(1)

temp_df = temp_df.dropna().reset_index(drop=True)

print("After feature engineering:", temp_df.shape)
temp_df.head()

After feature engineering: (129999, 25)


,uv_index,latitude,humidity,pressure_mb,air_quality_Ozone,condition_text,air_quality_Nitrogen_dioxide,cloud,visibility_km,longitude,...,temperature_celsius,temp_lag1,temp_lag2,temp_lag3,temp_roll3_mean,temp_roll5_mean,humidity_lag1,pressure_lag1,wind_lag1,cloud_lag1
0,1.0,17.25,89,1007.0,34.0,30,0.3,94,10.0,-88.77,...,26.0,20.0,26.0,23.0,24.000000,22.22,88.0,1019.0,13.6,100.0
1,1.0,12.15,80,1009.0,14.3,41,6.5,75,10.0,-86.27,...,27.2,26.0,20.0,26.0,24.400000,24.44,89.0,1007.0,4.3,94.0
2,1.0,9.97,100,1016.0,0.0,4,10.9,75,7.0,-84.08,...,21.0,27.2,26.0,20.0,24.733333,24.04,80.0,1009.0,3.6,75.0
3,1.0,19.43,47,1013.0,7.4,2,48.7,5,10.0,-99.13,...,20.8,21.0,27.2,26.0,23.000000,23.00,100.0,1016.0,2.2,75.0
4,1.0,17.97,84,1013.0,0.2,41,12.2,84,10.0,-76.75,...,21.9,20.8,21.0,27.2,21.233333,23.38,47.0,1013.0,6.7,5.0


In [ ]:
selected_features = [col for col in pm25_df.columns if col != target]

print("Number of features:", len(selected_features))
print(selected_features)

Number of features: 15
['uv_index', 'latitude', 'humidity', 'pressure_mb', 'air_quality_Ozone', 'condition_text', 'air_quality_Nitrogen_dioxide', 'cloud', 'visibility_km', 'longitude', 'air_quality_PM10', 'wind_mph', 'gust_mph', 'wind_degree', 'precip_mm']


In [ ]:
split_index = int(0.8 * len(temp_df))

train_df = temp_df.iloc[:split_index].copy()
test_df = temp_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (103999, 25)
Test shape : (26000, 25)


In [ ]:
from river import linear_model

online_model = compose.Pipeline(
    preprocessing.StandardScaler(),
    linear_model.LinearRegression()
)

model_name = "River Online Only"
print("Using model:", model_name)

Using model: River Online Only


In [ ]:
train_mse = metrics.MSE()
train_rmse = metrics.RMSE()
train_mae = metrics.MAE()
train_r2 = metrics.R2()

train_true = []
train_pred = []

for _, row in train_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    # predict current instance
    y_pred = online_model.predict_one(x)
    if y_pred is None:
        y_pred = 0.0

    train_true.append(y)
    train_pred.append(y_pred)

    train_mse.update(y, y_pred)
    train_rmse.update(y, y_pred)
    train_mae.update(y, y_pred)
    train_r2.update(y, y_pred)

    # learn from the same instance
    online_model.learn_one(x, y)

print("\nOnline Learning - Training Stream Results")
print("MSE :", round(train_mse.get(), 4))
print("RMSE:", round(train_rmse.get(), 4))
print("MAE :", round(train_mae.get(), 4))
print("R2  :", round(train_r2.get(), 4))
print("Accuracy (%):", round(train_r2.get() * 100, 2))


Online Learning - Training Stream Results
MSE : 23781598.063
RMSE: 4876.638
MAE : 250.7366
R2  : -303657.7976
Accuracy (%): -30365779.76


In [ ]:
online_train_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(train_mse.get(), 3),
    "RMSE": round(train_rmse.get(), 3),
    "MAE": round(train_mae.get(), 3),
    "R2": round(train_r2.get(), 3),
    "Accuracy (%)": round(train_r2.get() * 100, 3)
}])

online_train_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River Online Only,2.378160e+07,4876.638,250.737,-303657.798,-3.036578e+07


In [ ]:
test_mse = metrics.MSE()
test_rmse = metrics.RMSE()
test_mae = metrics.MAE()
test_r2 = metrics.R2()

test_true = []
test_pred = []

for _, row in test_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    y_hat = online_model.predict_one(x)
    if y_hat is None:
        y_hat = 0.0

    test_true.append(y)
    test_pred.append(y_hat)

    test_mse.update(y, y_hat)
    test_rmse.update(y, y_hat)
    test_mae.update(y, y_hat)
    test_r2.update(y, y_hat)

print("\nOnline Learning - Final Test Results")
print("MSE :", round(test_mse.get(), 4))
print("RMSE:", round(test_rmse.get(), 4))
print("MAE :", round(test_mae.get(), 4))
print("R2  :", round(test_r2.get(), 4))
print("Accuracy (%):", round(test_r2.get() * 100, 2))


Online Learning - Final Test Results
MSE : 4117.8323
RMSE: 64.1703
MAE : 46.6438
R2  : -32.1542
Accuracy (%): -3215.42


In [ ]:
online_test_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(test_mse.get(), 3),
    "RMSE": round(test_rmse.get(), 3),
    "MAE": round(test_mae.get(), 3),
    "R2": round(test_r2.get(), 3),
    "Accuracy (%)": round(test_r2.get() * 100, 3)
}])

online_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River Online Only,4117.832,64.17,46.644,-32.154,-3215.421


In [ ]:
from google.colab import files

online_train_df.to_csv("online_only_temperature_training_results.csv", index=False)
online_test_df.to_csv("online_only_temperature_test_results.csv", index=False)

files.download("online_only_temperature_training_results.csv")
files.download("online_only_temperature_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>